In [ ]:
!pip install librosa pandas scikit_learn gradio


In [ ]:
import os
import librosa
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib

In [ ]:
def extract_features_from_array(y, sr):
  mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)
  chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=sr), axis=1)
  contrast = np.mean(librosa.feature.spectral_contrast(y=y, sr=sr), axis=1)
  zcr = np.mean(librosa.feature.zero_crossing_rate(y))
  return np.hstack([mfccs, chroma, contrast, zcr])

In [ ]:
def augment_audio(y, sr):
    y_stretch = librosa.effects.time_stretch(y, rate=1.1)
    y_pitch = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=2)
    noise = 0.005 * np.random.randn(len(y))
    y_noise = y + noise
    y_stretch_pitch = librosa.effects.pitch_shift(y=y_stretch, sr=sr, n_steps=2)
    y_stretch_noise = y_stretch + 0.005 * np.random.randn(len(y_stretch))
    return [y, y_stretch, y_pitch, y_noise, y_stretch_pitch, y_stretch_noise]

In [ ]:
def process_file(file_path, emotion, truth):
    try:
        y, sr = librosa.load(file_path)
        augmented_versions = augment_audio(y, sr)
        data = []
        for y_aug in augmented_versions:
            features = extract_features_from_array(y_aug, sr)
            row = list(features) + [emotion, truth]
            data.append(row)
        return pd.DataFrame(data, columns=[f'feat_{i}' for i in range(len(data[0])-2)] + ['emotion', 'truth'])
    except Exception as e:
        print(f"❌ خطأ في الملف: {file_path} — {e}")
        return pd.DataFrame()

In [ ]:
import os
print(os.listdir("/content"))

In [ ]:
import zipfile

with zipfile.ZipFile("/content/data.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/data")

In [ ]:
base_dir = "/content/data/data"
df_all = pd.DataFrame()

for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if os.path.isdir(folder_path):
        try:
            emotion, truth = folder.split("_")
        except:
            print(f"⚠️ تجاهل المجلد غير الصحيح: {folder}")
            continue
        for file in os.listdir(folder_path):
            if file.endswith(".wav"):
                file_path = os.path.join(folder_path, file)
                df = process_file(file_path, emotion, truth)
                df_all = pd.concat([df_all, df], ignore_index=True)

print(f"✅ تم جمع البيانات بنجاح: {len(df_all)} عينة")

# حفظ البيانات فقط إذا كانت غير فارغة
if len(df_all) > 0:
    df_all.to_csv("dataset.csv", index=False)
else:
    print("⚠️ لم يتم حفظ الملف لأن البيانات فارغة")

In [ ]:
# تحميل البيانات
df_all = pd.read_csv("dataset.csv")

# فصل الميزات عن الأهداف
X = df_all.drop(['emotion', 'truth'], axis=1)
y_emotion = df_all['emotion']
y_truth = df_all['truth']

# تقسيم البيانات
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(X, y_emotion, test_size=0.2, stratify=y_emotion)
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X, y_truth, test_size=0.2, stratify=y_truth)

# نماذج المشاعر
models_emotion = {
    "RandomForest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "LogisticRegression": LogisticRegression(max_iter=1000)
}

# نماذج الحقيقة
models_truth = {
    "RandomForest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "LogisticRegression": LogisticRegression(max_iter=1000)
}

# تدريب وتقييم نماذج المشاعر
for name, model in models_emotion.items():
    model.fit(X_train_e, y_train_e)
    print(f"🎯 {name} - Emotion")
    print(classification_report(y_test_e, model.predict(X_test_e), zero_division=0))

# تدريب وتقييم نماذج الحقيقة
for name, model in models_truth.items():
    model.fit(X_train_t, y_train_t)
    print(f"🧠 {name} - Truth")
    print(classification_report(y_test_t, model.predict(X_test_t), zero_division=0))
    # حساب دقة كل نموذج للمشاعر
scores_emotion = {name: model.score(X_test_e, y_test_e) for name, model in models_emotion.items()}
best_emotion_model_name = max(scores_emotion, key=scores_emotion.get)
best_emotion_model = models_emotion[best_emotion_model_name]

# حساب دقة كل نموذج للحقيقة
scores_truth = {name: model.score(X_test_t, y_test_t) for name, model in models_truth.items()}
best_truth_model_name = max(scores_truth, key=scores_truth.get)
best_truth_model = models_truth[best_truth_model_name]

print(f"\n✅ أفضل نموذج للمشاعر: {best_emotion_model_name}")
print(f"✅ أفضل نموذج للحقيقة: {best_truth_model_name}")

In [ ]:
import joblib

joblib.dump(best_emotion_model, "emotion_model.pkl")
joblib.dump(best_truth_model, "truth_model.pkl")

print("💾 تم حفظ النموذجين بنجاح: emotion_model.pkl و truth_model.pkl")

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import librosa
import numpy as np
import joblib

# تحميل النموذجين
emotion_model = joblib.load("emotion_model.pkl")
truth_model = joblib.load("truth_model.pkl")

# دالة استخراج الميزات
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=None)
    mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).T, axis=0)
    chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=sr).T, axis=0)
    contrast = np.mean(librosa.feature.spectral_contrast(y=y, sr=sr).T, axis=0)
    zcr = np.mean(librosa.feature.zero_crossing_rate(y).T, axis=0)
    return np.hstack([mfcc, chroma, contrast, zcr])

# دالة التنبؤ
def predict_emotion_and_truth(audio):
    if audio is None:
        return "لم يتم تسجيل صوت", ""
    file_path = audio  # Gradio يعيد مسار الملف المؤقت
    features = extract_features(file_path)
    features = features.reshape(1, -1)
    emotion = emotion_model.predict(features)[0]
    truth = truth_model.predict(features)[0]
    return f"🎭 المشاعر: {emotion}", f"🧠 الحقيقة: {truth}"

# واجهة Gradio (تسجيل مباشر من الميكروفون)
with gr.Blocks() as demo:
    gr.Markdown("## 🎙️ تصنيف المشاعر وكشف الحقيقة من الصوت")
    gr.Markdown("سجّل صوتك وسنخبرك بالمشاعر وما إذا كانت حقيقية أو مزيفة")

    audio_input = gr.Audio(sources=["microphone"], type="filepath", label="سجّل صوتك هنا")
    emotion_output = gr.Textbox(label="المشاعر")
    truth_output = gr.Textbox(label="الحقيقة")

    btn = gr.Button("🔍 تحليل الصوت")
    btn.click(fn=predict_emotion_and_truth, inputs=audio_input, outputs=[emotion_output, truth_output])

demo.launch()

In [ ]:
def extract_features(file_path):
    y, sr = librosa.load(file_path)
    mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)
    chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=sr), axis=1)
    contrast = np.mean(librosa.feature.spectral_contrast(y=y, sr=sr), axis=1)
    zcr = np.mean(librosa.feature.zero_crossing_rate(y), axis=1)
    features = np.hstack([mfccs, chroma, contrast, zcr])
    return features.reshape(1, -1)

In [ ]:
def predict_emotion_and_truth(audio_file):
    try:
        features = extract_features(audio_file)
        emotion = emotion_model.predict(features)[0]
        truth = truth_model.predict(features)[0]
        return f"🎯 المشاعر: {emotion}", f"🧠 الحقيقة: {truth}"
    except Exception as e:
        return f"❌ خطأ في تصنيف المشاعر: {str(e)}", f"❌ خطأ في كشف الحقيقة"

In [ ]:
import gradio as gr

interface = gr.Interface(
    fn=predict_emotion_and_truth,
    inputs=gr.Audio(type="filepath", label="🎙️ ارفع ملف صوتي بصيغة WAV"),
    outputs=[
        gr.Textbox(label="🎯 تصنيف المشاعر"),
        gr.Textbox(label="🧠 كشف الحقيقة")
    ],
    title="🎛️ واجهة تصنيف المشاعر وكشف الحقيقة",
    description="ارفع ملف صوتي وسيتم تحليل المشاعر وكشف الحقيقة باستخدام النموذج المدرب"
)

interface.launch()